In [14]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
from itertools import combinations

articles = pd.read_csv("../../companies_in_articles.csv")
companies = pd.read_csv("../../nyse_nasdaq_companies_list.csv")

In [15]:
print(articles.columns)
print(companies.columns)

articles[["article_id", "company"]].head()

Index(['article_id', 'date', 'title', 'url', 'company'], dtype='str')
Index(['company_id', 'company', 'exchange', 'ticker', 'industries',
       'market_cap', 'market_cap_date', 'company_url', 'revenue',
       'revenue_date'],
      dtype='str')


,article_id,company
0,0,"['BBC', 'Honda', 'Renault', 'Red Bull', 'the R..."
1,1,"['GLOBE NEWSWIRE', 'ADDvantage Technologies Gr..."
2,2,"['FTSE', 'Helios Investment Partners', 'the Lo..."
3,3,['BRIEF-Pareteum Awarded']
4,4,"['TSX', '/PRNewswire/ - Jaguar Mining Inc', 'J..."


In [16]:
LEGAL_SUFFIXES = r"""
inc|inc\.|corporation|corp|corp\.|company|co|co\.|group|plc|
ltd|ltd\.|limited|holdings|holding|sa|s\.a\.|ag|nv|n\.v\.|
se|spa|s\.p\.a\.|asa|ab|gmbh|lp|llp|llc|l\.l\.c\.|bv|b\.v\.
"""

suffix_re = re.compile(rf"\b({LEGAL_SUFFIXES})\b", re.IGNORECASE | re.VERBOSE)

def normalize_name(x):
    if pd.isna(x):
        return None

    x = str(x).strip()
    if not x:
        return None

    # Apple Inc’s becomes Apple Inc
    x = x.replace("’s", "").replace("'s", "")
    x = x.replace("’", "'")

    # Reuters ticker style: AAPL.O becomes AAPL
    x = re.sub(r"\.[A-Z]+$", "", x)

    x = x.lower()
    x = x.replace("&", " and ")

    # We removed legal suffixes
    x = suffix_re.sub(" ", x)

    # We removed punctuation
    x = re.sub(r"[^a-z0-9 ]+", " ", x)

    # We collapse the spaces
    x = re.sub(r"\s+", " ", x).strip()

    return x or None

In [17]:
alias_rows = []

for _, row in companies.iterrows():
    company_id = row["company_id"]
    company_name = row["company"]

    for col in ["company", "ticker"]:


        value = row[col]

        if pd.isna(value):
            continue

        alias = normalize_name(value)

        if alias is None:
            continue

        if len(alias) < 3:
            continue

        alias_rows.append({
            "alias": alias,
            "company_id": company_id,
            "company": company_name,
            "source": col
        })

alias_df = pd.DataFrame(alias_rows).drop_duplicates()

alias_df.head()

,alias,company_id,company,source
0,buenaventura,Q1001788,Buenaventura,company
1,bvn,Q1001788,Buenaventura,ticker
2,build a bear workshop,Q1002992,Build-A-Bear Workshop,company
3,bbw,Q1002992,Build-A-Bear Workshop,ticker
4,united nuclear,Q100321332,United Nuclear Corporation,company


In [18]:
alias_counts = alias_df.groupby("alias")["company_id"].nunique()

ambiguous_aliases = set(alias_counts[alias_counts > 1].index)

alias_df_clean = alias_df[~alias_df["alias"].isin(ambiguous_aliases)].copy()

alias_to_company = dict(zip(alias_df_clean["alias"], alias_df_clean["company_id"]))

print("Aliases:", len(alias_df))
print("Ambiguous aliases removed:", len(ambiguous_aliases))
print("Clean aliases:", len(alias_to_company))

Aliases: 7792
Ambiguous aliases removed: 144
Clean aliases: 7343


In [6]:
def parse_company_list(x):
    if pd.isna(x):
        return []

    try:
        value = ast.literal_eval(x)
        if isinstance(value, list):
            return value
        return []
    except Exception:
        return []

articles["company_list"] = articles["company"].apply(parse_company_list)

articles[["article_id", "company_list"]].head()

,article_id,company_list
0,0,"[BBC, Honda, Renault, Red Bull, the Red Bull-o..."
1,1,"[GLOBE NEWSWIRE, ADDvantage Technologies Group..."
2,2,"[FTSE, Helios Investment Partners, the London ..."
3,3,[BRIEF-Pareteum Awarded]
4,4,"[TSX, /PRNewswire/ - Jaguar Mining Inc, JAG, C..."


In [7]:
def match_companies_in_article(company_names):
    matched = set()

    for name in company_names:
        alias = normalize_name(name)

        if alias in alias_to_company:
            matched.add(alias_to_company[alias])

    return matched

articles["matched_company_ids"] = articles["company_list"].apply(match_companies_in_article)

articles[["article_id", "company_list", "matched_company_ids"]].head()

,article_id,company_list,matched_company_ids
0,0,"[BBC, Honda, Renault, Red Bull, the Red Bull-o...",{Q9584}
1,1,"[GLOBE NEWSWIRE, ADDvantage Technologies Group...",{}
2,2,"[FTSE, Helios Investment Partners, the London ...","{Q154950, Q219508, Q372657}"
3,3,[BRIEF-Pareteum Awarded],{}
4,4,"[TSX, /PRNewswire/ - Jaguar Mining Inc, JAG, C...",{}


In [8]:
articles["n_matched_companies"] = articles["matched_company_ids"].apply(len)

print("Articles with at least 1 matched company:")
print((articles["n_matched_companies"] >= 1).sum())

print("Articles with at least 2 matched companies:")
print((articles["n_matched_companies"] >= 2).sum())

print("Unique matched companies:")
print(len(set().union(*articles["matched_company_ids"])))

Articles with at least 1 matched company:
17597
Articles with at least 2 matched companies:
6081
Unique matched companies:
2562


In [9]:
company_counts = Counter()

for company_ids in articles["matched_company_ids"]:
    company_counts.update(company_ids)

company_lookup = companies.set_index("company_id")["company"].to_dict()

company_count_df = pd.DataFrame([
    {
        "company_id": company_id,
        "company": company_lookup.get(company_id, company_id),
        "article_count": count
    }
    for company_id, count in company_counts.items()
])

company_count_df = company_count_df.sort_values("article_count", ascending=False)

company_count_df.head(50)

,company_id,company,article_count
34,Q312,Apple Inc.,684
38,Q3884,Amazon,587
59,Q95,Google,515
17,Q66,Boeing,434
56,Q193326,Goldman Sachs,409
73,Q334204,Morgan Stanley,394
33,Q2283,Microsoft,371
239,Q66048,Deutsche Bank,343
147,Q2529982,The New York Times Company,283
102,Q768773,MSCI,278


In [10]:
edge_counts = Counter()

for company_ids in articles["matched_company_ids"]:
    company_ids = sorted(company_ids)

    if len(company_ids) < 2:
        continue

    for a, b in combinations(company_ids, 2):
        edge_counts[(a, b)] += 1

edge_df = pd.DataFrame([
    {
        "source": a,
        "target": b,
        "weight": weight,
        "source_name": company_lookup.get(a, a),
        "target_name": company_lookup.get(b, b)
    }
    for (a, b), weight in edge_counts.items()
])

edge_df = edge_df.sort_values("weight", ascending=False)

edge_df.head(20)

,source,target,weight,source_name,target_name
99,Q312,Q95,126,Apple Inc.,Google
114,Q312,Q3884,119,Apple Inc.,Amazon
142,Q3884,Q95,108,Amazon,Google
70,Q20800404,Q95,108,Alphabet Inc.,Google
21,Q2283,Q312,88,Microsoft,Apple Inc.
151,Q2283,Q95,86,Microsoft,Google
65,Q193326,Q334204,69,Goldman Sachs,Morgan Stanley
242,Q2283,Q3884,66,Microsoft,Amazon
845,Q219508,Q334204,63,Citigroup,Morgan Stanley
25,Q3884,Q483551,61,Amazon,Walmart


In [11]:
company_count_df.to_csv("company_article_counts.csv", index=False)
edge_df.to_csv("company_comention_edges.csv", index=False)